In [0]:
pip install great_expectations

In [0]:
import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import regexp_extract
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.types import FloatType, TimestampType, StringType, IntegerType
import re
import great_expectations as gx

log_entries = []

def log_step(step, table, before, after, notes=''):
    entry = {
        'timestamp': datetime.datetime.utcnow().isoformat(),
        'stage': 'silver', 'step': step, 'table': table,
        'records_before': before, 'records_after': after,
        'dropped': before - after, 'notes': notes, 'status': 'success'
    }
    log_entries.append(entry)
    print(f'[{step}] {table}: {before} → {after} rows ({before-after} dropped)')

# Clean lung cancer data

In [0]:
df = spark.table('bronze.raw_lung_cancer')
before = df.count()
print(before)

In [0]:
for col_name in df.columns:
    df = df.withColumnRenamed(col_name, col_name.lower().replace(' ', '_'))
    
df.select("gender").distinct().show()
df.select("country").distinct().show()
df.select("age").describe().show()
for col in df.columns:
    print(col, df.select(col).first()[0])

In [0]:
str_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
for c in str_cols:
    df = df.withColumn(c, F.trim(F.col(c)))

df = df \
    .withColumn('age', F.col('age').cast(IntegerType())) \
    .withColumn("cancer_stage",
                F.when(regexp_extract("cancer_stage", r"\d+", 0) != "",
                       regexp_extract("cancer_stage", r"\d+", 0).cast("int"))
                .otherwise(None))\
    .withColumn('survival_years', F.col('survival_years').cast(DoubleType())) \
    .withColumn('cigarettes_per_day', F.col('cigarettes_per_day').cast(IntegerType()))

df = df.dropna(subset=['age', 'country', 'gender'])

df = df.dropDuplicates()

after = df.count()
log_step('clean_lung', 'silver.lung_cancer', before, after, 'cleaned + deduped')

In [0]:
df = df \
    .withColumn('_silver_timestamp', F.current_timestamp()) \
    .withColumn('_silver_version', F.lit('1.0'))

df.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('silver.lung_cancer')


# Clean WHO pollution data

In [0]:
df_who = spark.table('bronze.raw_who_pollution')
before_who = df_who.count()
print(before_who)
display(df_who.limit(10))

In [0]:
df_who = (
    spark.table("bronze.raw_who_pollution")
    .orderBy(F.col("_ingest_timestamp").desc())
    .dropDuplicates(["Year", "Country", "Gender", "Cause", "Dim2ValueCode", "Indicator"])
)

str_cols = [f.name for f in df_who.schema.fields if isinstance(f.dataType, StringType)]
for c in str_cols:
    df_who = df_who.withColumn(c, F.trim(F.col(c)))

df_who = (
    df_who
    .withColumn("Year",         F.col("Year").cast(IntegerType()))
    .withColumn("ValueNumeric", F.col("ValueNumeric").cast(FloatType()))
    .withColumn("ValueLow",     F.col("ValueLow").cast(FloatType()))
    .withColumn("ValueHigh",    F.col("ValueHigh").cast(FloatType()))
    .withColumn("Gender",
        F.when(F.upper(F.col("Gender")) == "BOTH SEXES", "Both")
         .otherwise(F.col("Gender"))
    )
    .withColumn("_silver_timestamp", F.current_timestamp())
    .withColumn("_silver_version",   F.lit("1.0"))
    .drop("Value")
)

df_who = (
    df_who
    .withColumn("Year",         F.when(F.col("Year").between(1900, 2100),  F.col("Year")))
    .withColumn("ValueNumeric", F.when(F.col("ValueNumeric") >= 0, F.col("ValueNumeric")))
    .withColumn("ValueLow",     F.when(F.col("ValueLow") >= 0,     F.col("ValueLow")))
    .withColumn("ValueHigh",    F.when(F.col("ValueHigh") >= 0,    F.col("ValueHigh")))
)





silver_table = "silver.who_pollution"

if spark.catalog.tableExists(silver_table):
    (DeltaTable.forName(spark, silver_table).alias("tgt")
        .merge(
            df_who.alias("src"),
            """tgt.Year          = src.Year AND
               tgt.Country       = src.Country AND
               tgt.Gender        = src.Gender AND
               tgt.Cause         = src.Cause AND
               tgt.Dim2ValueCode = src.Dim2ValueCode AND
               tgt.Indicator     = src.Indicator"""
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
else:
    df_who.write.format("delta").mode("overwrite").saveAsTable(silver_table)

print(f"Done: {df_who.count():,} records → {silver_table}")

# Clean air quality streaming data

In [0]:
df_aq = spark.table('bronze.raw_air_quality')
display(df_aq.limit(10))

In [0]:
df_aq = (
    spark.table("bronze.raw_air_quality")
    .orderBy(F.col("_fetched_at").desc())
    .dropDuplicates(["country", "time"])
)

def pm25_aqi(pm):
    return (
        F.when(pm <= 12.0, ((50-0)   / (12.0-0.0))    * (pm - 0.0)   + 0)
         .when(pm <= 35.4, ((100-51)  / (35.4-12.1))   * (pm - 12.1)  + 51)
         .when(pm <= 55.4, ((150-101) / (55.4-35.5))   * (pm - 35.5)  + 101)
         .when(pm <= 150.4,((200-151) / (150.4-55.5))  * (pm - 55.5)  + 151)
         .otherwise(        (300-201)  / (250.4-150.5)  * (pm - 150.5) + 201)
    )

def pm10_aqi(pm):
    return (
        F.when(pm <= 54,  ((50-0)   / (54-0))   * (pm - 0)   + 0)
         .when(pm <= 154, ((100-51)  / (154-55))  * (pm - 55)  + 51)
         .when(pm <= 254, ((150-101) / (254-155)) * (pm - 155) + 101)
         .when(pm <= 354, ((200-151) / (354-255)) * (pm - 255) + 151)
         .otherwise(       (300-201)  / (424-355)  * (pm - 355) + 201)
    )


df_aq = (
    df_aq
    .withColumn("aqi", F.round(F.greatest(pm25_aqi(F.col("pm2_5")), pm10_aqi(F.col("pm10"))), 1))
)

df_aq = (
    df_aq
    .groupBy("country")
    .agg(
        F.round(F.avg("pm10"),  2).alias("avg_pm10"),
        F.round(F.avg("pm2_5"), 2).alias("avg_pm2_5"),
        F.round(F.max("pm10"),  2).alias("max_pm10"),
        F.round(F.max("pm2_5"), 2).alias("max_pm2_5"),
        F.round(F.min("pm10"),  2).alias("min_pm10"),
        F.round(F.min("pm2_5"), 2).alias("min_pm2_5"),
        F.round(F.avg("aqi"),   1).alias("avg_aqi"),
        F.round(F.max("aqi"),   1).alias("max_aqi"),
        F.min("time").alias("period_start"),
        F.max("time").alias("period_end"),
        F.count("*").alias("reading_count"),
    )
    .withColumn("aqi_category",
        F.when(F.col("avg_aqi") <= 50,  "Good")
         .when(F.col("avg_aqi") <= 100, "Moderate")
         .when(F.col("avg_aqi") <= 150, "Unhealthy for Sensitive Groups")
         .when(F.col("avg_aqi") <= 200, "Unhealthy")
         .when(F.col("avg_aqi") <= 300, "Very Unhealthy")
         .otherwise("Hazardous")
    )
    .withColumn("_silver_timestamp", F.current_timestamp())
    .withColumn("_silver_version",   F.lit("1.0"))
)



In [0]:
silver_table = "silver.air_quality"

if spark.catalog.tableExists(silver_table):
    (DeltaTable.forName(spark, silver_table).alias("tgt")
        .merge(df_aq.alias("src"), "tgt.country = src.country")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
else:
    df_aq.write.format("delta").mode("overwrite").saveAsTable(silver_table)

print(f"Done: {df_aq.count():,} records → {silver_table}")

# Clean Pubmed Abstracts

In [ ]:
df_ab = (
    spark.table("bronze.raw_pubmed_keywords")
    .dropDuplicates(["first_line"])
)
display(df_ab.limit(20))

In [ ]:
def get_paragraphs(raw):
    import re
    return re.split(r'\n\n', (raw or '').strip())

@F.udf(IntegerType())
def get_year(raw):
    import re
    citation = get_paragraphs(raw)[0]
    m = re.search(r'\b(19|20)\d{2}\b', citation)
    return int(m.group()) if m else None

@F.udf(StringType())
def get_journal(raw):
    import re
    citation = get_paragraphs(raw)[0].replace('\n', ' ')
    m = re.match(r'^\d+\.\s+(.*?)\.\s+\d{4}', citation)
    return m.group(1).strip() if m else None

@F.udf(StringType())
def get_title(raw):
    parts = get_paragraphs(raw)
    return parts[1].replace('\n', ' ').strip() if len(parts) > 1 else None

@F.udf(StringType())
def get_authors(raw):
    import re
    parts = get_paragraphs(raw)
    authors_raw = parts[2].replace('\n', ' ').strip() if len(parts) > 2 else None
    if authors_raw and re.search(r'\w+\s+\w+\(\d+\)', authors_raw):
        return re.sub(r'\(\d+\)', '', authors_raw).strip()
    return None

@F.udf(StringType())
def get_abstract(raw):
    import re
    parts = get_paragraphs(raw)
    author_idx = next((i for i, p in enumerate(parts) if p.strip().startswith('Author information:')), None)
    if author_idx is None:
        return None
    abstract_parts = []
    for p in parts[author_idx + 1:]:
        if re.match(r'^(Copyright|©|DOI:|PMID:|PMCID:|Conflict)', p.strip()):
            break
        abstract_parts.append(p.replace('\n', ' ').strip())
    return ' '.join(abstract_parts).strip() or None

@F.udf(StringType())
def get_doi(raw):
    m = re.search(r'DOI:\s*(\S+)', raw or '')
    return m.group(1).rstrip('.') if m else None

@F.udf(StringType())
def get_pmid(raw):
    import re
    m = re.search(r'PMID:\s*(\d+)', raw or '')
    return m.group(1) if m else None

@F.udf(StringType())
def get_pmcid(raw):
    import re
    m = re.search(r'PMCID:\s*(\S+)', raw or '')
    return m.group(1) if m else None

df_ab = (
    df_ab
    .withColumn("year",     get_year("raw_text"))
    .withColumn("journal",  get_journal("raw_text"))
    .withColumn("title",    get_title("raw_text"))
    .withColumn("authors",  get_authors("raw_text"))
    .withColumn("abstract", get_abstract("raw_text"))
    .withColumn("doi",      get_doi("raw_text"))
    .withColumn("pmid",     get_pmid("raw_text"))
    .withColumn("pmcid",    get_pmcid("raw_text"))
    .select(
        "pmid", "pmcid", "year", "journal", "title",
        "authors", "abstract", "doi", "raw_text",
        "risk_keywords", "keyword_count",
        F.current_timestamp().alias("_silver_timestamp"),
        F.lit("1.0").alias("_silver_version")
    )
)

In [ ]:
silver_table = "silver.abstracts"

if spark.catalog.tableExists(silver_table):
    (DeltaTable.forName(spark, silver_table).alias("tgt")
        .merge(df_ab.alias("src"), "tgt.pmid = src.pmid")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
else:
    df_ab.write.format("delta").mode("overwrite").saveAsTable(silver_table)

print(f"Done: {df_ab.count():,} records → {silver_table}")

# Advanced Metadata and Validation

In [0]:
df_sample = spark.table('silver.lung_cancer').limit(10000).toPandas()
context = gx.get_context()
ds = context.data_sources.add_pandas(name="pandas_ds")
data_asset = ds.add_dataframe_asset(name="lung_cancer_sample")
batch_request = data_asset.build_batch_request(options={'dataframe': df_sample})

suite_name = 'lung_cancer_suite'
try:
    suite = context.suites.get(suite_name)
except:
    suite = context.suites.add(gx.ExpectationSuite(name=suite_name))

validator = context.get_validator(batch_request=batch_request, expectation_suite_name=suite_name)

validator.expect_column_to_exist('age')
validator.expect_column_values_to_not_be_null('country')
validator.expect_column_values_to_be_between('age', min_value=0, max_value=120)
validator.expect_column_values_to_be_in_set('gender', ['Male','Female'])
validator.expect_column_to_exist('cancer_stage')

results = validator.validate()
print(f'Validation passed: {results.success}')
print(f'Passed: {results.statistics["successful_expectations"]} / {results.statistics["evaluated_expectations"]}')